# LightGBM, one model per segment

Same as `lightgbm_training.ipynb`, but one model is tuned and trained on apartment-style homes (`is_apartment = 1`) and another on everything else (`is_apartment = 0`). Each home is predicted by the model of its segment and the two sets of predictions are combined at the end.

Features are still built on every sale, so community medians and lags see the whole market; only the model fitting is split.

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping
import optuna
import sys
from pathlib import Path
from dataclasses import replace
import warnings

DIR_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(DIR_ROOT))

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

In [ ]:
SEED = 987
TRACK = "off_market"                     # "off_market" or "list_aware"
VAL_START = pd.Timestamp("2025-12-01")    # sales on or after this date are validation
N_TRIALS = 100                             # optuna trials per segment
ES_FRACTION = 0.0                         # latest share of each fold's training rows used for early stopping
K_FOLDS = 5                              # CV validates on each of the last K_FOLDS months before VAL_START
META_MONTHS = 6                          # out-of-fold predictions for the meta model also cover the last META_MONTHS months before VAL_START
MODEL_NAME = "lightgbm_split_apartment"  # file name in oof_preds/<TRACK>/, change it for each model you want in the ensemble
DEBUG = True                             # print MdAPE and MAPE for every fold
SEGMENTS = {"apartment": 1, "house": 0}  # segment name -> is_apartment value

In [ ]:
import data

listings = data.load("listings")
off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [ ]:
from xxbonkxx_features import off_market_features, list_aware_features, make_is_apartment

make_features = off_market_features if TRACK == "off_market" else list_aware_features

train = listings[listings["date_sold"] < VAL_START]
val = listings[listings["date_sold"] >= VAL_START]

y_train = np.log(train["sold_price"])
y_val = np.log(val["sold_price"])

is_apartment = make_is_apartment(listings)

# Column names only. The matrices are built inside each fold, then again on train and val after tuning.
print(f"Features: {make_features(train.iloc[:10]).columns.tolist()}")
print(f"train: {len(train):,} sales up to {VAL_START.date()} ({is_apartment[train.index].sum():,} apartments)")
print(f"val:   {len(val):,} sales from {VAL_START.date()} ({is_apartment[val.index].sum():,} apartments)")

In [ ]:
def mdape_log(y_log, pred_log):
    return 100 * np.median(np.abs(np.exp(pred_log) / np.exp(y_log) - 1))

def mape_log(y_log, pred_log):
    return 100 * np.mean(np.abs(np.exp(pred_log) / np.exp(y_log) - 1))

def mdape(y, pred):
    return 100 * np.median(np.abs(pred - y) / np.abs(y))

def mape(y, pred):
    return 100 * np.mean(np.abs(pred - y) / np.abs(y))

target_space = "log"

In [ ]:
from training_utils import walk_forward_folds

CATEGORICAL_FEATURES = [
        "community",
        "house_style",
        "house_type",
    ]


def cv_months(K, end=VAL_START):
    """The K calendar months just before `end`, oldest first."""
    return pd.period_range(end=(end - pd.Timedelta(days=1)).to_period("M"), periods=K, freq="M")


def segment_fold(fold, is_apartment):
    """The part of a walk-forward fold whose homes have the given is_apartment value, in
    both the training sales and the predicted month. The is_apartment columns (and their
    interactions) are constant within a segment, so they are dropped."""
    in_train = fold.X_train["is_apartment"].astype(int) == is_apartment
    in_homes = fold.X_homes["is_apartment"].astype(int) == is_apartment
    apartment_columns = [col for col in fold.X_train.columns if col.startswith("is_apartment")]
    return replace(
        fold,
        homes=fold.homes.loc[in_homes[in_homes].index],
        X_train=fold.X_train.loc[in_train].drop(columns=apartment_columns),
        y_train=fold.y_train.loc[in_train],
        X_homes=fold.X_homes.loc[in_homes].drop(columns=apartment_columns),
    )


def cross_val(folds, y, params, debug=False, target_space="log"):
    """Monthly walk-forward CV, the way `python run.py validate` works: each fold
    predicts one month's sales (answers hidden) with a model trained on every earlier sale.
    `folds` comes from walk_forward_folds, so features are built once, not per trial.
    `y` is indexed like the homes the folds were built from. Early stopping uses the
    latest ES_FRACTION of each fold's training rows, never the validated month.
    Returns (mean validation loss, mean boosting rounds used) over the folds."""
    mdape_scores = []
    mape_scores = []
    rounds = []
    for fold in folds:
        X_tr, X_va = fold.X_train, fold.X_homes
        n_fit = int(len(X_tr) * (1 - ES_FRACTION))
        X_fit, X_es = X_tr.iloc[:n_fit], X_tr.iloc[n_fit:]
        y_fit, y_es, y_va = y.loc[X_fit.index], y.loc[X_es.index], y.loc[X_va.index]

        model = LGBMRegressor(**params, random_state=SEED, verbosity=-1)
        model.fit(X_fit, y_fit, categorical_feature=CATEGORICAL_FEATURES)
        rounds.append(model.best_iteration_)
        pred = model.predict(X_va)
        if target_space == "log":
            mdape_scores.append(mdape_log(y_va, pred))
            mape_scores.append(mape_log(y_va, pred))
        else:
            mdape_scores.append(mdape(y_va, pred))
            mape_scores.append(mape(y_va, pred))

        if debug:
            mdape_error = mdape_scores[-1]
            mape_error = mape_scores[-1]
            print(f"{fold.month}: train {len(X_fit):,}  early stop {len(X_es):,}  val {len(X_va):,}  "
                  f"rounds {rounds[-1]}  MdAPE {mdape_error:.2f}%  MAPE {mape_error:.2f}%", flush=True)

    print(f"mean MdAPE {np.mean(mdape_scores):.2f}%  mean MAPE {np.mean(mape_scores):.2f}%", flush=True)
    return np.mean(mdape_scores), int(np.mean(rounds))

In [ ]:
# Built once and shared by every trial of both segments: the last K_FOLDS months before VAL_START.
cv_folds = walk_forward_folds(train, make_features, TRACK, months=cv_months(K_FOLDS))
print("CV months:", ", ".join(str(fold.month) for fold in cv_folds))

segment_cv_folds = {
    segment: [segment_fold(fold, flag) for fold in cv_folds]
    for segment, flag in SEGMENTS.items()
}

def make_objective(folds):
    def objective(trial):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255),
            "max_depth": trial.suggest_int("max_depth", 3, 13),
            "n_estimators": trial.suggest_int("n_estimators", 200, 2000, step=200),
            "subsample_freq": 1,  # LightGBM ignores subsample unless bagging runs every k rounds
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0, step=0.1),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "min_split_gain": trial.suggest_float("min_split_gain", 0, 0.1, step=0.01),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0, step=0.1),
            "metric": "mae",
            "device": "gpu"
        }
        score, n_rounds = cross_val(folds, y_train, params, debug=DEBUG, target_space=target_space)
        trial.set_user_attr("n_rounds", n_rounds)
        return score
    return objective

In [ ]:
studies = {}
for segment, flag in SEGMENTS.items():
    print(f"\n===== {segment} (is_apartment = {flag}) =====")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        study_name=f"lightgbm_{segment}",
    )
    study.optimize(make_objective(segment_cv_folds[segment]), n_trials=N_TRIALS)
    studies[segment] = study

for segment, study in studies.items():
    print(f"\n{segment}")
    print("  best CV MdAPE:", round(study.best_value, 2))
    print("  best params:", study.best_params)
    print("  boosting rounds:", study.best_trial.user_attrs["n_rounds"])

In [ ]:
from optuna.importance import get_param_importances

for segment, study in studies.items():
    print(f"\n{segment}")
    for param, score in get_param_importances(study).items():
        print(f"  {param}: {score:.4f}")

In [ ]:
from training_utils import walk_forward_validate

def make_fit_model(params):
    def fit_model(X_train, y_train):
        model = LGBMRegressor(**params,
                              subsample_freq=1,
                              importance_type="gain",
                              random_state=SEED,
                              verbosity=-1)
        return model.fit(X_train, y_train)
    return fit_model

fit_models = {segment: make_fit_model(study.best_params) for segment, study in studies.items()}

folds = walk_forward_folds(listings, make_features, TRACK)
validations = {}
for segment, flag in SEGMENTS.items():
    print(f"\n===== {segment} (is_apartment = {flag}) =====")
    segment_folds = [segment_fold(fold, flag) for fold in folds]
    validations[segment] = walk_forward_validate(
        listings, make_features, fit_models[segment], TRACK, folds=segment_folds
    )

In [ ]:
for segment, validation in validations.items():
    feature_importances = pd.Series(
        validation.model.feature_importances_, index=validation.X_train.columns
    ).sort_values(ascending=False)
    print(f"===== {segment} =====")
    print("Most Important Features:")
    print(feature_importances.head(20))
    print("\n")
    print("Least Important Features:")
    print(feature_importances.tail(20))
    print("\n")

In [ ]:
# Combine: every validation home was predicted by the model of its own segment.
predictions = pd.concat([validation.predictions for validation in validations.values()])
predictions = predictions.sort_values(["month", "ml_num"])
X_val = pd.concat([fold.X_homes for fold in folds]).loc[predictions.index]
actual, predicted = predictions["sold_price"], predictions["predicted_price"]

def print_errors(label, actual, predicted):
    error = 100 * np.abs(predicted / actual - 1)
    print(f"{label:<10} {len(error):>7,} homes  MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%")

for segment, validation in validations.items():
    print_errors(segment, validation.actual, validation.predicted)
print_errors("combined", actual, predicted)

print("\nCombined, by month:")
for month, group in predictions.groupby("month"):
    print_errors(month, group["sold_price"], group["predicted_price"])

In [ ]:
# Out-of-fold predictions for the meta model (meta_training.ipynb). The META_MONTHS months
# before VAL_START are predicted the same walk-forward way as the validation months, each by
# its segment's model trained only on earlier sales, and saved together with the validation predictions.
from training_utils import save_oof_predictions

meta_folds = walk_forward_folds(listings, make_features, TRACK, months=cv_months(META_MONTHS))
print("Meta model training months:", ", ".join(str(fold.month) for fold in meta_folds))

meta_validations = {}
for segment, flag in SEGMENTS.items():
    print(f"\n===== {segment} (is_apartment = {flag}) =====")
    segment_folds = [segment_fold(fold, flag) for fold in meta_folds]
    meta_validations[segment] = walk_forward_validate(
        listings, make_features, fit_models[segment], TRACK, folds=segment_folds
    )

save_oof_predictions(MODEL_NAME, *meta_validations.values(), *validations.values())

In [ ]:
import diagnostics

diagnostics.plot_residuals_vs_actual(actual, predicted)
diagnostics.plot_predicted_vs_actual(actual, predicted)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_residuals_by_feature(X, actual, predicted, max_categories=20, n_cols=3):
    """One panel per column of X: a scatter plot for continuous columns and a box plot
    for categorical ones. Numeric columns with at most max_categories distinct values
    (bedrooms, month, ...) count as categorical. Columns with more categories than that
    show only their max_categories most common values."""
    residual = percent_error(actual, predicted)
    n_rows = int(np.ceil(len(X.columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)

    for ax, column in zip(axes.flat, X.columns):
        values = X[column].to_numpy()
        is_numeric = pd.api.types.is_numeric_dtype(X[column])
        if is_numeric and X[column].nunique() > max_categories:
            ax.scatter(values, residual, s=3, alpha=0.3)
        else:
            if is_numeric:
                order = sorted(X[column].value_counts().index[:max_categories])
            else:
                order = (
                    pd.DataFrame({"category": X[column], "residual": residual})
                    .groupby("category")["residual"]
                    .mean()
                    .sort_values(ascending=False)
                    .head(max_categories)
                    .index
                    .tolist()
                )
            sns.boxplot(x=values, y=residual, order=order, showfliers=False, ax=ax)
            ax.tick_params(axis="x", rotation=90)
        ax.axhline(0, color="black", linewidth=1)
        ax.set(title=column, xlabel="", ylabel="Residual (%)")

    for ax in axes.flat[len(X.columns):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()

def percent_error(actual, predicted):
    return 100 * (np.asarray(predicted) / np.asarray(actual) - 1)

plot_residuals_by_feature(X_val, actual, predicted)

In [ ]:
def evaluate_by_quantile(y_true, y_pred, n_bins=10):
    df = pd.DataFrame({
        "actual": y_true,
        "predicted": y_pred
    })
    df["quantile"] = pd.qcut(
        df["actual"], q=n_bins, duplicates="drop"
    )
    df["residual"] = df["actual"] - df["predicted"]
    df["residual_percent"] = 100 * np.abs(df["residual"]) / df["actual"]
    return df.groupby("quantile", observed=True).agg(
        count=("actual", "size"),
        MAE=("residual", lambda x: x.abs().mean()),
        mean_residual=("residual", "mean"),
        mean_residual_percent=("residual_percent", "mean"),
        mdape=("residual_percent", lambda x: np.median(np.abs(x))),
        mape=("residual_percent", lambda x: np.mean(np.abs(x)))
    )

evaluate_by_quantile(actual, predicted, n_bins=20)